# Validación computacional de resultados: duración de la titulación en el pregrado universitario chileno (2025)
Sumativa 2, Fase 3 · MCDI501 · Integrantes: Fernanda Ovalle Román, Sebastián Cajales Cid, César Lorca Bacián, Jorge Álvarez Ossandón.

Este notebook valida, mediante simulación y remuestreo, los resultados obtenidos en la Sumativa 1 (`S1/notebooks/Sumativa1_F2_Titulados.ipynb`): intervalos de confianza, pruebas de hipótesis y correlaciones. Usa la misma fuente de datos y la misma preparación.

Fuente: Titulados de Educación Superior 2025 (Mineduc). El archivo crudo debe estar en `data/raw/` en la raíz del repositorio.

## 1. Preparación de datos (replicada de la Sumativa 1)
Se repite exactamente la preparación de la S1 para trabajar sobre la misma base: carga del crudo, filtro a pregrado universitario, códigos de relleno (1900 y 0) convertidos a NaN sin imputar, variables derivadas (`semestres_reales`, `atraso_semestres`, `edad_titulacion`) y marcas de atípicos (Tukey) e inconsistencias. Los atípicos se conservan; en la sección de robustez se evalúa su efecto.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Raíz del repo: el notebook vive en S2/notebooks/, así que subimos dos niveles
RAIZ = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
RUTA_CRUDO = RAIZ / "data" / "raw" / "20260817_Titulados_Ed_Superior_2025_WEB.csv"
RUTA_FIG = RAIZ / "S2" / "informe" / "figuras"
RUTA_RES = RAIZ / "S2" / "resultados"

crudo = pd.read_csv(RUTA_CRUDO, sep=";", encoding="utf-8", low_memory=False)
print(f"Filas: {crudo.shape[0]:,} | Columnas: {crudo.shape[1]}")

Filas: 328,998 | Columnas: 41


In [2]:
filtro = (crudo["nivel_global"] == "Pregrado") & (crudo["tipo_inst_1"] == "Universidades")
df = crudo.loc[filtro].copy()

# Códigos de relleno → NaN, con bandera, sin imputar
df["ingreso_sin_info"] = df["anio_ing_carr_ori"].eq(1900)
df["anio_ing_carr_ori"] = df["anio_ing_carr_ori"].replace(1900, np.nan)
df["sem_ing_carr_ori"]  = df["sem_ing_carr_ori"].replace(0, np.nan)

constantes = [col for col in df.columns if df[col].nunique(dropna=False) == 1]
df = df.drop(columns=constantes)

print(f"Pregrado U: {len(df):,} filas")
print(f"Sin info de ingreso: {df['ingreso_sin_info'].sum():,}")
print("Columnas constantes eliminadas:", constantes)

Pregrado U: 105,063 filas
Sin info de ingreso: 13,396
Columnas constantes eliminadas: ['cat_periodo', 'tipo_inst_1', 'nivel_global']


In [3]:
anio_tit = df["fecha_obtencion_titulo"] // 10000
mes_tit  = (df["fecha_obtencion_titulo"] // 100) % 100
sem_tit  = np.where(mes_tit <= 6, 1, 2)

# Semestres reales: del semestre de ingreso al de titulación, ambos incluidos
idx_ingreso = df["anio_ing_carr_ori"] * 2 + (df["sem_ing_carr_ori"] - 1)
idx_titulo  = anio_tit * 2 + (sem_tit - 1)
df["semestres_reales"] = idx_titulo - idx_ingreso + 1
df["atraso_semestres"] = df["semestres_reales"] - df["dur_total_carr"]

anio_nac = df["fec_nac_alu"] // 100
mes_nac  = df["fec_nac_alu"] % 100
df["edad_titulacion"] = ((anio_tit * 12 + mes_tit) - (anio_nac * 12 + mes_nac)) / 12

df["sexo"] = df["gen_alu"].map({1: "Hombre", 2: "Mujer"})
UMBRAL_ALTO = 4
df["atraso_alto"] = df["atraso_semestres"] > UMBRAL_ALTO

def marcar_atipicos(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    ric = q3 - q1
    return (serie < q1 - 1.5 * ric) | (serie > q3 + 1.5 * ric)

df["incons_antes_formal"] = df["semestres_reales"] < df["dur_total_carr"]
df["atipico_semestres"] = marcar_atipicos(df["semestres_reales"])
df["atipico_edad"]      = marcar_atipicos(df["edad_titulacion"])

print(df[["semestres_reales", "atraso_semestres", "edad_titulacion"]].agg(["count", "mean"]).round(3))
print(df[["incons_antes_formal", "atipico_semestres", "atipico_edad"]].sum())

       semestres_reales  atraso_semestres  edad_titulacion
count         91667.000         91667.000       105063.000
mean             12.986             3.398           27.966
incons_antes_formal     1511
atipico_semestres      13201
atipico_edad           10711
dtype: int64


## 2. Resultados de la Sumativa 1 a validar
Se registran los valores reportados en la S1 (secciones 2 a 4 de su notebook). Cada método de esta evaluación se compara contra ellos: IC *t* de la media (bootstrap), *t* de Welch y chi-cuadrado (permutación), correlaciones de Spearman (bootstrap) y parámetros de entrada para la simulación Monte Carlo.

In [4]:
# IC 95 % t para la media (S1, sección 3)
S1_IC_MEDIA = pd.DataFrame({
    "media":    [3.398, 12.986, 27.966],
    "IC95_inf": [3.367, 12.953, 27.927],
    "IC95_sup": [3.429, 13.019, 28.004],
}, index=["atraso_semestres", "semestres_reales", "edad_titulacion"])

# Prueba 1: atraso según sexo, t de Welch (S1, sección 4)
S1_WELCH = {"dif_medias": 1.068, "IC95_inf": 1.004, "IC95_sup": 1.132,
            "t": 32.76, "p": 1.09e-233, "d_cohen": 0.226,
            "media_h": 3.999, "sd_h": 5.434, "n_h": 40075,
            "media_m": 2.931, "sd_m": 4.101, "n_m": 51592}

# Prueba 2: jornada × atraso alto, chi-cuadrado (S1, sección 4)
S1_CHI2 = {"chi2": 316.09, "p": 1.03e-70, "V_cramer": 0.060,
           "pct_alto_diurna": 23.9, "pct_alto_vespertina": 32.9}

# Correlaciones de Spearman (S1, sección 2)
S1_SPEARMAN = pd.Series({
    ("semestres_reales", "atraso_semestres"): 0.82,
    ("atraso_semestres", "edad_titulacion"):  0.47,
    ("semestres_reales", "edad_titulacion"):  0.46,
    ("dur_total_carr",   "edad_titulacion"): -0.19,
    ("dur_total_carr",   "atraso_semestres"): 0.01,
}, name="rho_S1")

display(S1_IC_MEDIA)
display(pd.Series(S1_WELCH, name="Welch S1").to_frame().T)
display(pd.Series(S1_CHI2, name="Chi2 S1").to_frame().T)
display(S1_SPEARMAN.to_frame())

,media,IC95_inf,IC95_sup
atraso_semestres,3.398,3.367,3.429
semestres_reales,12.986,12.953,13.019
edad_titulacion,27.966,27.927,28.004


,dif_medias,IC95_inf,IC95_sup,t,p,d_cohen,media_h,sd_h,n_h,media_m,sd_m,n_m
Welch S1,1.068,1.004,1.132,32.76,1.090000e-233,0.226,3.999,5.434,40075.0,2.931,4.101,51592.0


,chi2,p,V_cramer,pct_alto_diurna,pct_alto_vespertina
Chi2 S1,316.09,1.030000e-70,0.06,23.9,32.9


rho_S1
semestres_reales atraso_semestres    0.82
atraso_semestres edad_titulacion     0.47
semestres_reales edad_titulacion     0.46
dur_total_carr   edad_titulacion    -0.19
                 atraso_semestres    0.01

In [5]:
# Verificación: recalcular IC t de la media y Welch con el df de esta S2
for v in S1_IC_MEDIA.index:
    x = df[v].dropna()
    ic = stats.t.interval(0.95, df=len(x) - 1, loc=x.mean(), scale=stats.sem(x))
    print(f"{v:18s} media={x.mean():.3f}  IC95=[{ic[0]:.3f}; {ic[1]:.3f}]")

h = df.loc[df["sexo"] == "Hombre", "atraso_semestres"].dropna()
m = df.loc[df["sexo"] == "Mujer",  "atraso_semestres"].dropna()
welch = stats.ttest_ind(h, m, equal_var=False)
print(f"\nWelch: dif={h.mean() - m.mean():.3f}  t={welch.statistic:.2f}  p={welch.pvalue:.2e}")

atraso_semestres   media=3.398  IC95=[3.367; 3.429]
semestres_reales   media=12.986  IC95=[12.953; 13.019]
edad_titulacion    media=27.966  IC95=[27.927; 28.004]

Welch: dif=1.068  t=32.76  p=1.09e-233
